In [2]:
# Prepare the Data for Machine Learning Algorithms
import pandas as pd
import numpy as np
from utils import load_housing_data
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from preprocessingUtils import getPreprocessing

preprocessing = getPreprocessing()
housing = load_housing_data()
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)


strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

# Revert to a clean training set
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()



In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

# Создаём pipeline:
# 1) preprocessing — превращает сырые данные (housing) в 24 числовые фичи
# 2) LinearRegression — обучает модель на этих фичах
lin_reg = make_pipeline(preprocessing, LinearRegression())


# Обучение модели:
# housing — входные данные (признаки районов)
# housing_labels — реальные цены (что нужно предсказать)
# Модель учится находить зависимость: признаки → цена
lin_reg.fit(housing, housing_labels)


# Делаем предсказания на тех же данных (train set)
# Для каждого района считаем его предполагаемую цену
housing_predictions = lin_reg.predict(housing)


# Выводим первые 5 предсказаний
# round(-2) — округление до сотен (чтобы проще читать)
print(
    "predict -\n",
    housing_predictions[:5].round(-2)
)


# Выводим реальные значения (цены) для этих же 5 районов
print(
    "real - \n",
    housing_labels.iloc[:5].values
)

predict -
 [246000. 372700. 135700.  91400. 330900.]
real - 
 [458300. 483800. 101700.  96100. 361800.]


In [5]:
# Measure RMSE 
from sklearn.metrics import root_mean_squared_error
lin_rmse = root_mean_squared_error(housing_labels, housing_predictions)
lin_rmse

68972.88910758478

In [ ]:
# Try DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)
housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels, housing_predictions)
print(tree_rmse)

0.0


In [ ]:
# Cross validation (считаем ошибку на сколько хорошо)
# k-fold кросс-валидация даёт более надёжную оценку модели:
# train делится на k частей, модель обучается k раз и проверяется на каждой части
from sklearn.model_selection import cross_val_score
tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
scoring="neg_root_mean_squared_error", cv=10)

pd.Series(tree_rmses).describe()

count       10.000000
mean     67013.360949
std       1460.198570
min      64289.376198
25%      66776.146282
50%      67086.216281
75%      68140.275029
max      68659.294290
dtype: float64

In [ ]:
# Try a RandomForestRegressor
# Random Forest значительно лучше: RMSE ~47k vs ~68k у линейной и дерева
# → более точные и стабильные предсказания
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(
    forest_reg, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=10
)
pd.Series(forest_rmses).describe()

count       10.000000
mean     47124.604437
std       1069.311372
min      45292.329302
25%      46712.106520
50%      47172.209883
75%      47561.377695
max      49354.705514
dtype: float64

In [ ]:
# Grid Search
# поиск оптимального гиперпараметра
# searches for the best combination of hyperparameter values for the RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

full_pipeline = Pipeline(
    [
        ("preprocessing", preprocessing),
        ("random_forest", RandomForestRegressor(random_state=42)),
    ]
)
param_grid = [
    {
		# try count of cluster like 5, 8, 10 instead default 10
        "preprocessing__geo__n_clusters": [5, 8, 10],
        "random_forest__max_features": [4, 6, 8],
		# 3 × 3 = 9 вариантов
    },
    {
        "preprocessing__geo__n_clusters": [10, 15],
        "random_forest__max_features": [6, 8, 10],
		# 2 × 3 = 6 вариантов
    },
]
# GridSearchCV → перебирает параметры
grid_search = GridSearchCV(
	# neg_root_mean_squared_errorКаждую комбинацию 
    # он проверяет через 3-fold cross-validation train dataset [Часть A] [Часть B] [Часть C]
	# 15 × 3 = 45 обучений модели 
    full_pipeline, param_grid, cv=3, scoring="neg_root_mean_squared_error"
)

# 1️⃣ Первый прогон
# train: B + C
# test: A
# 2️⃣ Второй прогон
# train: A + C
# test: B
# 3️⃣ Третий прогон
# train: A + B
# test: C
grid_search.fit(housing, housing_labels)
# be a {'preprocessing__geo__n_clusters': 15, 'random_forest__max_features': 6}
grid_search.best_params_

{'preprocessing__geo__n_clusters': 15, 'random_forest__max_features': 6}

In [5]:
# evaluation scores
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False, inplace=True)
cv_res.head() # note: the 1st column is the row ID

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_preprocessing__geo__n_clusters,param_random_forest__max_features,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
12,4.577027,0.086923,0.112430,0.000239,15,6,"{'preprocessing__geo__n_clusters': 15, 'random...",-43520.694854,-43958.237963,-44759.841846,-44079.591555,513.105796,1
13,5.890098,0.044892,0.108793,0.002671,15,8,"{'preprocessing__geo__n_clusters': 15, 'random...",-44027.285408,-44179.627865,-45007.255144,-44404.722806,430.570141,2
14,7.207827,0.236511,0.117907,0.005450,15,10,"{'preprocessing__geo__n_clusters': 15, 'random...",-44402.775080,-44618.770051,-45423.947846,-44815.164326,439.413615,3
7,4.955483,0.115381,0.127067,0.006230,10,6,"{'preprocessing__geo__n_clusters': 10, 'random...",-44250.873441,-44825.226423,-45610.878897,-44895.659587,557.449151,4
9,5.075543,0.074072,0.129500,0.002498,10,6,"{'preprocessing__geo__n_clusters': 10, 'random...",-44250.873441,-44825.226423,-45610.878897,-44895.659587,557.449151,4


In [6]:
# Randomized Search

# grid search is fine for few combinations, but RandomizedSearchCV is often preferable
# especially when the hyperparameter search space is large
# It's evaluates a fixed number of combinations, selecting a random value for each
# hyperparametr at every iteration

# For each hyperparameter, you must provide either a list of possible values, or a
# probability distribution:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distribs = {
    "preprocessing__geo__n_clusters": randint(low=3, high=50),
    "random_forest__max_features": randint(low=2, high=20),
}
rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distribs,
	# попробовать 10 случайных комбинаций
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
)
rnd_search.fit(housing, housing_labels)

# Analyzing the Best Models and Their Errors
final_model = rnd_search.best_estimator_ # includes preprocessing
feature_importances = final_model["random_forest"].feature_importances_
feature_importances.round(2)


array([0.06, 0.06, 0.05, 0.01, 0.01, 0.01, 0.01, 0.19, 0.01, 0.02, 0.01,
       0.02, 0.01, 0.  , 0.01, 0.01, 0.01, 0.02, 0.01, 0.  , 0.01, 0.01,
       0.01, 0.  , 0.01, 0.  , 0.02, 0.01, 0.01, 0.  , 0.01, 0.01, 0.01,
       0.02, 0.02, 0.01, 0.01, 0.01, 0.04, 0.01, 0.02, 0.01, 0.01, 0.01,
       0.02, 0.01, 0.  , 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.  , 0.08,
       0.  , 0.  , 0.  , 0.01])